In [ ]:
# Notebook 10
# Physics-Informed Feature Engineering

**Project:** Machine Learning-Based Prediction of Degradation in PEM Fuel Cells Using Time-Series Operational Data

This notebook develops physically meaningful features from the processed PEMFC durability dataset for subsequent feature selection and predictive modelling.

The engineered features are derived from measured pressure, temperature, humidification, flow, electrical, and temporal variables. Each candidate feature is evaluated before retention to ensure that it is physically interpretable, mathematically valid, free from target leakage, sufficiently variable, not unnecessarily redundant, and potentially relevant to voltage prediction or durability behaviour.

Feature engineering is therefore treated as a hypothesis-driven process rather than the automatic generation of additional variables.

In [ ]:
## Objectives

The main objectives of this notebook are to:

- construct physically meaningful candidate features;
- verify the numerical validity of each engineered feature;
- assess overall and durability-stage variation;
- evaluate relationships with the voltage target;
- assess redundancy with original parent variables;
- identify features suitable for subsequent feature selection and modelling.

In [ ]:
## Notebook Workflow

The notebook follows the sequence:

1. Environment and dataset setup
2. Core feature-engineering requirements
3. Pressure-derived features
4. Temperature-derived features
5. Dew-point and humidification-related features
6. Flow and reactant-related features
7. Temporal and dynamic features
8. Engineered-feature validation
9. Target association assessment
10. Redundancy assessment
11. Feature carry-forward decisions
12. Save feature-engineered dataset
13. Feature Engineering Summary

In [ ]:
# Import Required Libraries

The libraries required for numerical processing, tabular data manipulation, visualisation, and statistical analysis are imported below.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from scipy.stats import pearsonr, spearmanr
from sklearn.feature_selection import mutual_info_regression

In [ ]:
# Notebook Configuration

Basic display settings are configured to make numerical outputs and tables easier to inspect throughout the notebook.

In [2]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

print("Notebook configuration completed.")

Notebook configuration completed.


In [ ]:
# Define Project Paths

The main project directories are defined so that the notebook can consistently access processed data, reusable source code, figures, and results.

In [3]:
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DATA_DIR = DATA_DIR / "processed"

SRC_DIR = PROJECT_ROOT / "src"
FIGURES_DIR = PROJECT_ROOT / "figures"
RESULTS_DIR = PROJECT_ROOT / "results"

print("Project root:", PROJECT_ROOT)
print("Processed data:", PROCESSED_DATA_DIR)
print("Source directory:", SRC_DIR)
print("Figures directory:", FIGURES_DIR)
print("Results directory:", RESULTS_DIR)

Project root: C:\Users\usman\Desktop\PEMFC_Dissertation
Processed data: C:\Users\usman\Desktop\PEMFC_Dissertation\data\processed
Source directory: C:\Users\usman\Desktop\PEMFC_Dissertation\src
Figures directory: C:\Users\usman\Desktop\PEMFC_Dissertation\figures
Results directory: C:\Users\usman\Desktop\PEMFC_Dissertation\results


In [ ]:
# Configure Project Source Package

The project root is added to the Python path so that reusable functions stored in the `src` package can be imported when required.

In [4]:
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("Project source package configured successfully.")

Project source package configured successfully.


In [ ]:
# Load Processed Dataset

The cleaned and analysis-ready PEMFC dataset produced during the earlier preprocessing stages is loaded as the starting point for feature engineering.

A separate working copy is created so that the original processed dataset remains unchanged.

In [5]:
processed_file = PROCESSED_DATA_DIR / "operational_cleaned.csv"

df = pd.read_csv(processed_file)

# Separate working copy for feature engineering
df_fe = df.copy()

print("Processed dataset loaded successfully.")
print("Dataset shape:", df_fe.shape)

Processed dataset loaded successfully.
Dataset shape: (3629680, 18)


In [ ]:
# Verify Dataset

Before engineering new features, the loaded dataset is checked to confirm its dimensions, columns, data types, and basic data integrity.

In [6]:
print("Dataset shape:")
print(df_fe.shape)

print("\nColumns:")
print(df_fe.columns.tolist())

print("\nData types:")
display(df_fe.dtypes.to_frame("dtype"))

Dataset shape:
(3629680, 18)

Columns:
['operating_hour', 'time', 'current', 'voltage', 'power', 'pressure_anode_inlet', 'pressure_anode_outlet', 'pressure_cathode_inlet', 'pressure_cathode_outlet', 'temp_anode_endplate', 'temp_anode_dewpoint_water', 'temp_anode_inlet', 'temp_anode_outlet', 'temp_cathode_dewpoint_water', 'temp_cathode_inlet', 'temp_cathode_outlet', 'total_anode_stack_flow', 'total_cathode_stack_flow']

Data types:


,dtype
operating_hour,int64
time,float64
current,float64
voltage,float64
power,float64
pressure_anode_inlet,float64
pressure_anode_outlet,float64
pressure_cathode_inlet,float64
pressure_cathode_outlet,float64
temp_anode_endplate,float64


In [ ]:
## Basic Integrity Check

Missing values, duplicate observations, and infinite numerical values are checked before any new features are constructed.

In [7]:
print("Total missing values:", df_fe.isna().sum().sum())

print("Duplicate rows:", df_fe.duplicated().sum())

numeric_df = df_fe.select_dtypes(include=np.number)

print(
    "Infinite numerical values:",
    np.isinf(numeric_df).sum().sum()
)

Total missing values: 0
Duplicate rows: 0
Infinite numerical values: 0


In [ ]:
# Preview Dataset

The first few observations are displayed to verify that the processed dataset has loaded correctly and to inspect its structure before feature engineering.

In [9]:
df_fe.head()

,operating_hour,time,current,voltage,power,pressure_anode_inlet,pressure_anode_outlet,pressure_cathode_inlet,pressure_cathode_outlet,temp_anode_endplate,temp_anode_dewpoint_water,temp_anode_inlet,temp_anode_outlet,temp_cathode_dewpoint_water,temp_cathode_inlet,temp_cathode_outlet,total_anode_stack_flow,total_cathode_stack_flow
0,50,1.7610,0.0000,0.9375,0.0000,109.9013,110.3273,109.8001,108.7921,83.1282,54.4647,71.9515,39.4532,64.4643,69.6736,56.3375,0.0700,0.2910
1,50,2.7610,0.0000,0.9375,0.0000,110.1037,110.3273,109.8001,108.8934,83.0788,54.4647,71.9021,39.3870,64.4390,69.6612,56.3249,0.0700,0.2910
2,50,3.7610,0.0000,0.9372,0.0000,110.3060,110.3273,109.8001,108.7921,83.0788,54.5293,71.8897,39.4135,64.4866,69.6859,56.2997,0.0700,0.2910
3,50,4.7610,0.0000,0.9375,0.0000,110.1037,110.3273,109.8001,108.7921,83.1035,54.4777,71.9391,39.3870,64.4390,69.6859,56.3249,0.0700,0.2910
4,50,5.7610,0.0000,0.9372,0.0000,109.9013,110.3273,109.6990,108.8934,83.0911,54.4777,71.8897,39.3738,64.4895,69.6612,56.2492,0.0700,0.2910


In [ ]:
# Core Feature Engineering Requirements

Every candidate engineered feature will be assessed using the same core requirements before it is retained.

A feature should:

1. have a clear physical interpretation;
2. be mathematically valid;
3. avoid direct or indirect target leakage;
4. contain useful variation;
5. avoid unnecessary redundancy;
6. have plausible predictive or durability-related relevance.

Passing the theoretical assessment allows a feature to be implemented as a candidate. Final retention will depend on empirical evidence from the dataset.

In [ ]:
## Feature Evaluation Sequence

Each candidate feature will be evaluated using the following sequence:

**Feature construction**

↓

**Numerical validity**

↓

**Overall distribution**

↓

**Durability-stage behaviour**

↓

**Voltage association**

↓

**Parent-feature redundancy**

↓

**Carry-forward decision**

This ensures that physically meaningful features are not automatically retained unless they also provide useful information in the observed dataset.

In [ ]:
## Required Variables

Before feature construction begins, the variables required for the initially proposed physics-informed features are checked.

The first feature group requires anode and cathode inlet/outlet pressures, inlet/outlet temperatures, dew-point temperatures, voltage, and durability-stage information.

In [8]:
required_columns = [
    "voltage",

    "pressure_anode_inlet",
    "pressure_anode_outlet",
    "pressure_cathode_inlet",
    "pressure_cathode_outlet",

    "temp_anode_inlet",
    "temp_anode_outlet",
    "temp_cathode_inlet",
    "temp_cathode_outlet",

    "temp_anode_dewpoint_water",
    "temp_cathode_dewpoint_water"
]

missing_columns = [
    column
    for column in required_columns
    if column not in df_fe.columns
]

if missing_columns:
    print("Missing required columns:")
    print(missing_columns)
else:
    print("All required variables are available.")

All required variables are available.


In [ ]:
# 10.1 Pressure-Derived Features

Pressure-derived features combine related inlet and outlet measurements to represent subsystem pressure behaviour more directly than the individual measurements alone.

Each feature is created as a candidate and then evaluated for numerical validity, variation, durability-stage behaviour, target association, and redundancy.

In [ ]:
## 10.1.1 Anode inlet–outlet Pressure Difference

The Anode inlet–outlet Pressure Difference is defined as:

\[
\Delta P_a = P_{a,inlet} - P_{a,outlet}
\]

It represents the measured inlet–outlet pressure difference across the hydrogen-side gas path.

In [ ]:
### Create Feature

The feature is calculated from the anode inlet and outlet pressure measurements using a consistent inlet-minus-outlet sign convention.

In [11]:
df_fe["anode_pressure_diff"] = (
    df_fe["pressure_anode_inlet"]
    - df_fe["pressure_anode_outlet"]
)

print("Feature created: anode_pressure_diff")

Feature created: anode_pressure_diff


In [ ]:
### Preview Feature

The first few values are inspected to confirm that the feature has been calculated correctly.

In [12]:
df_fe[
    [
        "pressure_anode_inlet",
        "pressure_anode_outlet",
        "anode_pressure_diff"
    ]
].head()

,pressure_anode_inlet,pressure_anode_outlet,anode_pressure_diff
0,109.9013,110.3273,-0.4259
1,110.1037,110.3273,-0.2236
2,110.3060,110.3273,-0.0212
3,110.1037,110.3273,-0.2236
4,109.9013,110.3273,-0.4259


In [ ]:
### Numerical Validity

The engineered feature is checked for missing, infinite, zero, and negative values before further analysis.

In [13]:
feature = "anode_pressure_diff"

validity_summary = pd.Series({
    "count": df_fe[feature].count(),
    "missing": df_fe[feature].isna().sum(),
    "infinite": np.isinf(df_fe[feature]).sum(),
    "zero": (df_fe[feature] == 0).sum(),
    "negative": (df_fe[feature] < 0).sum()
})

validity_summary

count       3629680
missing           0
infinite          0
zero              0
negative    2795606
dtype: int64

In [ ]:
### Overall Distribution

The overall distribution of the anode pressure difference is summarised using minimum, maximum, median, interquartile range (IQR), and standard deviation (SD).

These statistics determine whether the feature contains meaningful variation and help assess the magnitude and direction of the observed pressure differences.

In [14]:
q1 = df_fe["anode_pressure_diff"].quantile(0.25)
q3 = df_fe["anode_pressure_diff"].quantile(0.75)

anode_pressure_summary = pd.Series({
    "Min": df_fe["anode_pressure_diff"].min(),
    "Max": df_fe["anode_pressure_diff"].max(),
    "Median": df_fe["anode_pressure_diff"].median(),
    "IQR": q3 - q1,
    "SD": df_fe["anode_pressure_diff"].std()
})

anode_pressure_summary.round(4)

Min      -1.4542
Max      14.9511
Median   -0.2238
IQR       0.5057
SD        0.4191
dtype: float64

In [ ]:
### Durability-Stage Behaviour

The anode pressure difference is summarised separately for each durability stage.

Stage-wise median, IQR, and standard deviation are examined to determine whether the feature's central tendency or variability changes as operating hours increase.

In [20]:
anode_pressure_stage_summary = (
    df_fe
    .groupby("operating_hour")["anode_pressure_diff"]
    .agg(
        Median="median",
        Q1=lambda x: x.quantile(0.25),
        Q3=lambda x: x.quantile(0.75),
        SD="std"
    )
)

anode_pressure_stage_summary["IQR"] = (
    anode_pressure_stage_summary["Q3"]
    - anode_pressure_stage_summary["Q1"]
)

anode_pressure_stage_summary = (
    anode_pressure_stage_summary[
        ["Median", "Q1", "Q3", "IQR", "SD"]
    ]
    .round(4)
)

anode_pressure_stage_summary

,Median,Q1,Q3,IQR,SD
operating_hour,,,,,
50,-0.1224,-0.2237,0.1808,0.4046,0.3496
100,-0.4258,-0.5270,-0.1226,0.4044,0.3387
150,-0.3246,-0.5270,-0.1224,0.4046,0.3549
200,0.0797,-0.1226,0.3829,0.5055,0.4354
250,-0.4258,-0.5277,-0.2232,0.3045,0.3592
300,-0.2236,-0.4259,0.0797,0.5057,0.3969
350,-0.4258,-0.5271,-0.2229,0.3043,0.3659
400,-0.0214,-0.2236,0.2820,0.5056,0.4597
450,-0.2237,-0.5268,-0.0215,0.5053,0.3840


In [ ]:
### Visualise Durability-Stage Behaviour

The stage-wise median anode pressure difference is visualised together with its interquartile range (IQR).

The median represents the typical pressure difference at each durability stage, while the shaded IQR region represents the middle 50% of observations. All durability stages are displayed to support direct comparison across the full test period.

In [21]:
plt.figure(figsize=(12, 5))

# Median
plt.plot(
    anode_pressure_stage_summary.index,
    anode_pressure_stage_summary["Median"],
    marker="o",
    label="Stage median"
)

# IQR region
plt.fill_between(
    anode_pressure_stage_summary.index,
    anode_pressure_stage_summary["Q1"],
    anode_pressure_stage_summary["Q3"],
    alpha=0.2,
    label="Interquartile range (IQR)"
)

# Show every durability stage
plt.xticks(
    anode_pressure_stage_summary.index,
    rotation=45
)

plt.xlabel("Operating Hour")
plt.ylabel("Anode Pressure Difference")
plt.title(
    "Anode Pressure Difference Behaviour Across Durability Stages"
)

plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()

plt.show()

<Figure size 1200x500 with 1 Axes>

In [ ]:
### Visualise Stage-Wise Variability

The IQR and standard deviation are compared across durability stages to examine whether the variability of the anode pressure difference changes during the durability test.

In [22]:
plt.figure(figsize=(12, 5))

plt.plot(
    anode_pressure_stage_summary.index,
    anode_pressure_stage_summary["IQR"],
    marker="o",
    label="IQR"
)

plt.plot(
    anode_pressure_stage_summary.index,
    anode_pressure_stage_summary["SD"],
    marker="o",
    label="SD"
)

# Display all durability stages
plt.xticks(
    anode_pressure_stage_summary.index,
    rotation=45
)

plt.xlabel("Operating Hour")
plt.ylabel("Variability")
plt.title("Stage-Wise Variability of Anode Pressure Difference")

plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()

plt.show()

<Figure size 1200x500 with 1 Axes>

In [ ]:
### Voltage Association

The relationship between the anode pressure difference and voltage is evaluated using Pearson correlation, Spearman rank correlation, and Mutual Information.

These measures assess linear, monotonic, and more general statistical dependence respectively. Association with voltage indicates potential predictive relevance but does not by itself demonstrate degradation sensitivity or causality.

In [18]:
x = df_fe["anode_pressure_diff"]
y = df_fe["voltage"]

pearson_r, _ = pearsonr(x, y)
spearman_rho, _ = spearmanr(x, y)

mi = mutual_info_regression(
    df_fe[["anode_pressure_diff"]],
    y,
    random_state=42
)[0]

anode_pressure_voltage_association = pd.Series({
    "Pearson": pearson_r,
    "Spearman": spearman_rho,
    "Mutual Information": mi
})

anode_pressure_voltage_association.round(4)

Pearson              -0.6024
Spearman             -0.5305
Mutual Information    0.5600
dtype: float64

In [ ]:
### Parent-Feature Redundancy

The engineered anode pressure difference is compared with its original inlet and outlet pressure variables.

This determines whether the derived feature provides a distinct representation or largely reproduces information already contained in one of its parent variables.

In [19]:
parent_features = [
    "pressure_anode_inlet",
    "pressure_anode_outlet"
]

redundancy_results = []

for parent in parent_features:
    
    pearson_r, _ = pearsonr(
        df_fe["anode_pressure_diff"],
        df_fe[parent]
    )
    
    spearman_rho, _ = spearmanr(
        df_fe["anode_pressure_diff"],
        df_fe[parent]
    )
    
    redundancy_results.append({
        "Parent Feature": parent,
        "Pearson": pearson_r,
        "Spearman": spearman_rho
    })

anode_pressure_redundancy = pd.DataFrame(redundancy_results)

anode_pressure_redundancy.round(4)

,Parent Feature,Pearson,Spearman
0,pressure_anode_inlet,0.1067,0.4463
1,pressure_anode_outlet,-0.2270,-0.6529


In [ ]:
### Feature Decision

The anode pressure difference shows meaningful variation across the dataset and changes in both central tendency and variability across durability stages.

It also demonstrates moderate-to-strong association with voltage while not behaving as a simple linear duplicate of either parent pressure variable.

Therefore, `anode_pressure_diff` is retained as a candidate feature for subsequent feature selection and predictive modelling.

**Decision: Carry forward**

In [ ]:
## Feature 2 — Cathode Pressure Difference

The cathode pressure difference is defined as:

\[
\Delta P_c = P_{c,inlet} - P_{c,outlet}
\]

This feature represents the measured pressure difference across the cathode gas path.

The dataset documentation identifies these variables as the inlet and outlet
pressures of air. The inlet pressure was controlled around a specified operating
condition, while the recorded measurements may vary around the control value.

The engineered feature is investigated as a physically interpretable candidate
representation of cathode-side pressure behaviour. It is not assumed to directly
measure mass-transport limitation, flooding, or degradation.

The feature will be evaluated for:

- numerical validity;
- overall distribution and variability;
- behaviour across durability stages;
- association with voltage;
- redundancy with its parent pressure variables.

Its usefulness will be determined empirically before deciding whether it should
be retained for predictive modelling.

In [23]:
# Create cathode pressure difference
df_fe["cathode_pressure_diff"] = (
    df_fe["pressure_cathode_inlet"]
    - df_fe["pressure_cathode_outlet"]
)

print("Feature created: cathode_pressure_diff")

Feature created: cathode_pressure_diff


In [ ]:
### Verify Feature Construction

The first few observations are inspected alongside the original cathode inlet and outlet pressures to confirm that the engineered feature has been calculated correctly using:

\[
\Delta P_c = P_{c,inlet} - P_{c,outlet}
\]

In [24]:
df_fe[
    [
        "pressure_cathode_inlet",
        "pressure_cathode_outlet",
        "cathode_pressure_diff"
    ]
].head()

,pressure_cathode_inlet,pressure_cathode_outlet,cathode_pressure_diff
0,109.8001,108.7921,1.0080
1,109.8001,108.8934,0.9067
2,109.8001,108.7921,1.0080
3,109.8001,108.7921,1.0080
4,109.6990,108.8934,0.8056


In [ ]:
### Numerical Validity

The engineered cathode pressure difference is checked for missing, infinite, zero, and negative values.

This verifies that feature construction has not introduced invalid values and identifies the sign behaviour of the pressure difference across the complete dataset.

In [25]:
feature = "cathode_pressure_diff"

cathode_validity_summary = pd.Series({
    "count": df_fe[feature].count(),
    "missing": df_fe[feature].isna().sum(),
    "infinite": np.isinf(df_fe[feature]).sum(),
    "zero": (df_fe[feature] == 0).sum(),
    "negative": (df_fe[feature] < 0).sum()
})

cathode_validity_summary

count       3629680
missing           0
infinite          0
zero              0
negative         80
dtype: int64

In [ ]:
### Overall Distribution

The overall distribution of the cathode pressure difference is summarised using minimum, maximum, median, interquartile range (IQR), and standard deviation (SD).

These statistics are used to determine whether the engineered feature contains meaningful variation and to characterise its overall magnitude and spread.

In [26]:
q1 = df_fe["cathode_pressure_diff"].quantile(0.25)
q3 = df_fe["cathode_pressure_diff"].quantile(0.75)

cathode_pressure_summary = pd.Series({
    "Min": df_fe["cathode_pressure_diff"].min(),
    "Max": df_fe["cathode_pressure_diff"].max(),
    "Median": df_fe["cathode_pressure_diff"].median(),
    "IQR": q3 - q1,
    "SD": df_fe["cathode_pressure_diff"].std()
})

cathode_pressure_summary.round(4)

Min      -0.3073
Max      25.6125
Median    1.4125
IQR       1.6197
SD        1.7294
dtype: float64

In [ ]:
### Durability-Stage Behaviour

The cathode pressure difference is summarised separately at each durability stage using the median, first quartile (Q1), third quartile (Q3), interquartile range (IQR), and standard deviation (SD).

This allows changes in both central tendency and variability to be examined from 50 h to 1000 h.

In [27]:
cathode_pressure_stage_summary = (
    df_fe
    .groupby("operating_hour")["cathode_pressure_diff"]
    .agg(
        Median="median",
        Q1=lambda x: x.quantile(0.25),
        Q3=lambda x: x.quantile(0.75),
        SD="std"
    )
)

cathode_pressure_stage_summary["IQR"] = (
    cathode_pressure_stage_summary["Q3"]
    - cathode_pressure_stage_summary["Q1"]
)

cathode_pressure_stage_summary = (
    cathode_pressure_stage_summary[
        ["Median", "Q1", "Q3", "IQR", "SD"]
    ]
    .round(4)
)

cathode_pressure_stage_summary

,Median,Q1,Q3,IQR,SD
operating_hour,,,,,
50,1.4127,0.9075,2.6274,1.7199,1.7024
100,1.4127,0.9072,2.5273,1.6201,1.7063
150,1.3122,0.8057,2.5271,1.7214,1.7243
200,1.5138,1.0082,2.5275,1.5193,1.6821
250,1.4131,1.0082,2.5273,1.5191,1.6963
300,1.3126,0.9087,2.5252,1.6166,1.7042
350,1.4133,1.0082,2.5271,1.5189,1.7154
400,1.4127,1.0080,2.5273,1.5193,1.7249
450,1.4127,1.0082,2.5275,1.5193,1.7458


In [ ]:
### Visualise Durability-Stage Behaviour

The stage-wise median cathode pressure difference is visualised together with its interquartile range (IQR).

The median represents the typical pressure difference at each durability stage, while the shaded IQR region represents the middle 50% of observations. All durability stages are displayed for direct comparison across the test period.

In [28]:
plt.figure(figsize=(12, 5))

plt.plot(
    cathode_pressure_stage_summary.index,
    cathode_pressure_stage_summary["Median"],
    marker="o",
    label="Stage median"
)

plt.fill_between(
    cathode_pressure_stage_summary.index,
    cathode_pressure_stage_summary["Q1"],
    cathode_pressure_stage_summary["Q3"],
    alpha=0.2,
    label="Interquartile range (IQR)"
)

plt.xticks(
    cathode_pressure_stage_summary.index,
    rotation=45
)

plt.xlabel("Operating Hour")
plt.ylabel("Cathode Pressure Difference")
plt.title(
    "Cathode Pressure Difference Behaviour Across Durability Stages"
)

plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()

plt.show()

<Figure size 1200x500 with 1 Axes>

In [ ]:
### Visualise Stage-Wise Variability

The IQR and standard deviation are compared across durability stages to examine whether the variability of the cathode pressure difference changes during the durability test.

In [29]:
plt.figure(figsize=(12, 5))

plt.plot(
    cathode_pressure_stage_summary.index,
    cathode_pressure_stage_summary["IQR"],
    marker="o",
    label="IQR"
)

plt.plot(
    cathode_pressure_stage_summary.index,
    cathode_pressure_stage_summary["SD"],
    marker="o",
    label="SD"
)

plt.xticks(
    cathode_pressure_stage_summary.index,
    rotation=45
)

plt.xlabel("Operating Hour")
plt.ylabel("Variability")
plt.title(
    "Stage-Wise Variability of Cathode Pressure Difference"
)

plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()

plt.show()

<Figure size 1200x500 with 1 Axes>

In [ ]:
### Voltage Association

The relationship between the cathode pressure difference and voltage is evaluated using Pearson correlation, Spearman rank correlation, and Mutual Information.

Together, these measures assess linear, monotonic, and more general statistical dependence with the prediction target.

Association with voltage indicates potential predictive relevance, but does not by itself demonstrate degradation sensitivity or causality.

In [30]:
x = df_fe["cathode_pressure_diff"]
y = df_fe["voltage"]

pearson_r, _ = pearsonr(x, y)
spearman_rho, _ = spearmanr(x, y)

mi = mutual_info_regression(
    df_fe[["cathode_pressure_diff"]],
    y,
    random_state=42
)[0]

cathode_pressure_voltage_association = pd.Series({
    "Pearson": pearson_r,
    "Spearman": spearman_rho,
    "Mutual Information": mi
})

cathode_pressure_voltage_association.round(4)

Pearson              -0.8570
Spearman             -0.8181
Mutual Information    1.4286
dtype: float64

In [ ]:
### Parent-Feature Redundancy

The cathode pressure difference is compared with its original inlet and outlet pressure variables.

This assessment determines whether the engineered feature provides a distinct representation of cathode pressure behaviour or largely reproduces information already contained in one of its parent variables.

In [31]:
parent_features = [
    "pressure_cathode_inlet",
    "pressure_cathode_outlet"
]

redundancy_results = []

for parent in parent_features:

    pearson_r, _ = pearsonr(
        df_fe["cathode_pressure_diff"],
        df_fe[parent]
    )

    spearman_rho, _ = spearmanr(
        df_fe["cathode_pressure_diff"],
        df_fe[parent]
    )

    redundancy_results.append({
        "Parent Feature": parent,
        "Pearson": pearson_r,
        "Spearman": spearman_rho
    })

cathode_pressure_redundancy = pd.DataFrame(redundancy_results)

cathode_pressure_redundancy.round(4)

,Parent Feature,Pearson,Spearman
0,pressure_cathode_inlet,0.5163,0.6988
1,pressure_cathode_outlet,-0.6010,-0.4273


In [ ]:
## Feature 3 — Anode Temperature Difference

The anode temperature difference is defined as:

\[
\Delta T_a = T_{a,outlet} - T_{a,inlet}
\]

using:

- `temp_anode_inlet`
- `temp_anode_outlet`

This feature represents the measured temperature difference between the anode outlet and inlet.

It is investigated as a physically interpretable indicator of anode-side thermal behaviour. It is not assumed to directly represent heat generation, heat flux, thermal efficiency, or degradation.

The feature will be evaluated for:

- numerical validity;
- overall distribution and variability;
- behaviour across durability stages;
- association with voltage;
- redundancy with its parent temperature variables.

Its usefulness will be determined empirically before deciding whether it should be retained for predictive modelling.

In [32]:
# Create anode temperature difference
df_fe["anode_temp_diff"] = (
    df_fe["temp_anode_outlet"]
    - df_fe["temp_anode_inlet"]
)

print("Feature created: anode_temp_diff")

Feature created: anode_temp_diff


In [ ]:
### Verify Feature Construction

The first few observations are inspected alongside the original anode inlet and outlet temperatures to confirm that the engineered feature has been calculated correctly using:

\[
\Delta T_a = T_{a,outlet} - T_{a,inlet}
\]

Positive values indicate a higher measured outlet temperature, while negative values indicate a lower measured outlet temperature relative to the inlet.

In [33]:
df_fe[
    [
        "temp_anode_inlet",
        "temp_anode_outlet",
        "anode_temp_diff"
    ]
].head()

,temp_anode_inlet,temp_anode_outlet,anode_temp_diff
0,71.9515,39.4532,-32.4983
1,71.9021,39.3870,-32.5150
2,71.8897,39.4135,-32.4763
3,71.9391,39.3870,-32.5521
4,71.8897,39.3738,-32.5159


In [ ]:
### Numerical Validity

The engineered anode temperature difference is checked for missing, infinite, zero, and negative values.

This verifies that feature construction has not introduced invalid values and identifies the sign behaviour of the temperature difference across the complete dataset.

In [34]:
feature = "anode_temp_diff"

anode_temp_validity_summary = pd.Series({
    "count": df_fe[feature].count(),
    "missing": df_fe[feature].isna().sum(),
    "infinite": np.isinf(df_fe[feature]).sum(),
    "zero": (df_fe[feature] == 0).sum(),
    "negative": (df_fe[feature] < 0).sum()
})

anode_temp_validity_summary

count       3629680
missing           0
infinite          0
zero              0
negative    3629680
dtype: int64

In [ ]:
### Overall Distribution

The overall distribution of the anode temperature difference is summarised using minimum, maximum, median, interquartile range (IQR), and standard deviation (SD).

These statistics determine the typical magnitude and variability of the temperature difference and whether the engineered feature contains sufficient variation for further investigation.

In [35]:
q1 = df_fe["anode_temp_diff"].quantile(0.25)
q3 = df_fe["anode_temp_diff"].quantile(0.75)

anode_temp_summary = pd.Series({
    "Min": df_fe["anode_temp_diff"].min(),
    "Max": df_fe["anode_temp_diff"].max(),
    "Median": df_fe["anode_temp_diff"].median(),
    "IQR": q3 - q1,
    "SD": df_fe["anode_temp_diff"].std()
})

anode_temp_summary.round(4)

Min      -44.4706
Max      -12.0361
Median   -28.9556
IQR        7.9591
SD         5.6767
dtype: float64

In [ ]:
### Stage-Wise Behaviour

The anode temperature difference is summarised separately at each durability stage using the median, first quartile (Q1), third quartile (Q3), interquartile range (IQR), and standard deviation (SD).

This allows us to assess whether its central tendency or variability changes as operating hours increase.

In [36]:
anode_temp_stage_summary = (
    df_fe
    .groupby("operating_hour")["anode_temp_diff"]
    .agg(
        Median="median",
        Q1=lambda x: x.quantile(0.25),
        Q3=lambda x: x.quantile(0.75),
        SD="std"
    )
)

anode_temp_stage_summary["IQR"] = (
    anode_temp_stage_summary["Q3"]
    - anode_temp_stage_summary["Q1"]
)

anode_temp_stage_summary = anode_temp_stage_summary[
    ["Median", "Q1", "Q3", "IQR", "SD"]
]

anode_temp_stage_summary.round(4)

,Median,Q1,Q3,IQR,SD
operating_hour,,,,,
50,-26.5831,-28.3389,-24.3642,3.9747,2.7110
100,-32.9420,-36.1835,-31.2796,4.9039,3.1619
150,-30.3520,-32.6055,-28.7688,3.8367,2.5693
200,-23.3345,-27.2114,-21.5801,5.6313,5.4645
250,-34.0763,-35.8795,-30.8877,4.9918,3.5447
300,-31.3122,-36.6732,-25.4472,11.2260,6.0119
350,-32.1487,-34.6721,-30.6166,4.0555,2.5982
400,-30.7915,-35.0771,-23.1622,11.9150,6.4283
450,-27.0778,-32.6048,-25.8599,6.7449,3.8988


In [ ]:
### Visualise Durability-Stage Behaviour

The stage-wise median anode temperature difference is visualised together with its interquartile range (IQR).

The median represents the typical temperature difference at each durability stage, while the shaded IQR region represents the middle 50% of observations.

In [37]:
plt.figure(figsize=(12, 5))

plt.plot(
    anode_temp_stage_summary.index,
    anode_temp_stage_summary["Median"],
    marker="o",
    label="Stage median"
)

plt.fill_between(
    anode_temp_stage_summary.index,
    anode_temp_stage_summary["Q1"],
    anode_temp_stage_summary["Q3"],
    alpha=0.2,
    label="Interquartile range (IQR)"
)

plt.xticks(
    anode_temp_stage_summary.index,
    rotation=45
)

plt.xlabel("Operating Hour")
plt.ylabel("Anode Temperature Difference (°C)")
plt.title(
    "Anode Temperature Difference Behaviour Across Durability Stages"
)

plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()

plt.show()

<Figure size 1200x500 with 1 Axes>

In [ ]:
### Visualise Stage-Wise Variability

The IQR and standard deviation are compared across durability stages to examine whether the variability of the anode temperature difference changes during the durability test.

In [38]:
plt.figure(figsize=(12, 5))

plt.plot(
    anode_temp_stage_summary.index,
    anode_temp_stage_summary["IQR"],
    marker="o",
    label="IQR"
)

plt.plot(
    anode_temp_stage_summary.index,
    anode_temp_stage_summary["SD"],
    marker="o",
    label="SD"
)

plt.xticks(
    anode_temp_stage_summary.index,
    rotation=45
)

plt.xlabel("Operating Hour")
plt.ylabel("Variability (°C)")
plt.title(
    "Stage-Wise Variability of Anode Temperature Difference"
)

plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()

plt.show()

<Figure size 1200x500 with 1 Axes>

In [ ]:
### Voltage Association

The relationship between the anode temperature difference and voltage is evaluated using Pearson correlation, Spearman rank correlation, and Mutual Information.

These measures assess linear, monotonic, and more general statistical dependence with voltage.

Association with voltage indicates potential predictive relevance, but does not by itself demonstrate degradation sensitivity or causality.

In [39]:
x = df_fe["anode_temp_diff"]
y = df_fe["voltage"]

pearson_r, _ = pearsonr(x, y)
spearman_rho, _ = spearmanr(x, y)

mi = mutual_info_regression(
    df_fe[["anode_temp_diff"]],
    y,
    random_state=42
)[0]

anode_temp_voltage_association = pd.Series({
    "Pearson": pearson_r,
    "Spearman": spearman_rho,
    "Mutual Information": mi
})

anode_temp_voltage_association.round(4)

Pearson              -0.0104
Spearman             -0.0377
Mutual Information    0.2460
dtype: float64

In [ ]:
### Parent-Feature Redundancy

The anode temperature difference is compared with its original inlet and outlet temperature variables.

This assessment determines whether the engineered feature provides a distinct representation of anode thermal behaviour or largely reproduces information already contained in one of its parent variables.

In [40]:
parent_features = [
    "temp_anode_inlet",
    "temp_anode_outlet"
]

redundancy_results = []

for parent in parent_features:

    pearson_r, _ = pearsonr(
        df_fe["anode_temp_diff"],
        df_fe[parent]
    )

    spearman_rho, _ = spearmanr(
        df_fe["anode_temp_diff"],
        df_fe[parent]
    )

    redundancy_results.append({
        "Parent Feature": parent,
        "Pearson": pearson_r,
        "Spearman": spearman_rho
    })

anode_temp_redundancy = pd.DataFrame(redundancy_results)

anode_temp_redundancy.round(4)

,Parent Feature,Pearson,Spearman
0,temp_anode_inlet,0.0249,0.0390
1,temp_anode_outlet,0.9971,0.9968


In [ ]:
## Feature 4 — Cathode Temperature Change

The cathode inlet-to-outlet temperature change is defined as:

\[
\Delta T_c = T_{c,outlet} - T_{c,inlet}
\]

using:

- `temp_cathode_inlet`
- `temp_cathode_outlet`

This feature represents the measured change in cathode temperature from the inlet to the outlet.

Under this convention:

- positive values indicate that the outlet temperature is higher than the inlet temperature;
- negative values indicate that the outlet temperature is lower than the inlet temperature.

The feature is treated as a cathode-side thermal-response indicator and is not assumed to directly measure heat generation, water accumulation, mass-transport loss, or degradation.

Its numerical validity, variability, durability-stage behaviour, voltage association, and redundancy with its parent variables will be evaluated before determining its usefulness for predictive modelling.

In [41]:
# Create cathode inlet-to-outlet temperature change
df_fe["cathode_temp_diff"] = (
    df_fe["temp_cathode_outlet"]
    - df_fe["temp_cathode_inlet"]
)

print("Feature created: cathode_temp_diff")

Feature created: cathode_temp_diff


In [ ]:
### Verify Feature Construction

The first few observations are inspected alongside the original cathode inlet and outlet temperatures to confirm that the engineered feature has been calculated correctly using:

\[
\Delta T_c = T_{c,outlet} - T_{c,inlet}
\]

The sign indicates the direction of the measured inlet-to-outlet temperature change.

In [42]:
df_fe[
    [
        "temp_cathode_inlet",
        "temp_cathode_outlet",
        "cathode_temp_diff"
    ]
].head()

,temp_cathode_inlet,temp_cathode_outlet,cathode_temp_diff
0,69.6736,56.3375,-13.3360
1,69.6612,56.3249,-13.3363
2,69.6859,56.2997,-13.3863
3,69.6859,56.3249,-13.3610
4,69.6612,56.2492,-13.4121


In [ ]:
### Numerical Validity

The engineered cathode temperature change is checked for missing, infinite, zero, and negative values.

This verifies that feature construction has not introduced invalid values and identifies the sign behaviour of the temperature change across the complete dataset.

In [43]:
feature = "cathode_temp_diff"

cathode_temp_validity_summary = pd.Series({
    "count": df_fe[feature].count(),
    "missing": df_fe[feature].isna().sum(),
    "infinite": np.isinf(df_fe[feature]).sum(),
    "zero": (df_fe[feature] == 0).sum(),
    "negative": (df_fe[feature] < 0).sum()
})

cathode_temp_validity_summary

count       3629680
missing           0
infinite          0
zero              0
negative    3629680
dtype: int64

In [ ]:
### Overall Distribution

The overall distribution of the cathode temperature change is summarised using minimum, maximum, median, interquartile range (IQR), and standard deviation (SD).

These statistics describe the typical magnitude and variability of the engineered feature and determine whether it contains sufficient variation for further investigation.

In [44]:
q1 = df_fe["cathode_temp_diff"].quantile(0.25)
q3 = df_fe["cathode_temp_diff"].quantile(0.75)

cathode_temp_summary = pd.Series({
    "Min": df_fe["cathode_temp_diff"].min(),
    "Max": df_fe["cathode_temp_diff"].max(),
    "Median": df_fe["cathode_temp_diff"].median(),
    "IQR": q3 - q1,
    "SD": df_fe["cathode_temp_diff"].std()
})

cathode_temp_summary.round(4)

Min      -24.1436
Max       -2.5720
Median   -14.0870
IQR        4.0158
SD         2.8911
dtype: float64

In [ ]:
### Stage-Wise Behaviour

The cathode temperature change is summarised separately at each durability stage using the median, first quartile (Q1), third quartile (Q3), interquartile range (IQR), and standard deviation (SD).

This allows us to assess whether its typical behaviour or variability changes as operating hours increase.

In [45]:
cathode_temp_stage_summary = (
    df_fe
    .groupby("operating_hour")["cathode_temp_diff"]
    .agg(
        Median="median",
        Q1=lambda x: x.quantile(0.25),
        Q3=lambda x: x.quantile(0.75),
        SD="std"
    )
)

cathode_temp_stage_summary["IQR"] = (
    cathode_temp_stage_summary["Q3"]
    - cathode_temp_stage_summary["Q1"]
)

cathode_temp_stage_summary = cathode_temp_stage_summary[
    ["Median", "Q1", "Q3", "IQR", "SD"]
]

cathode_temp_stage_summary.round(4)

,Median,Q1,Q3,IQR,SD
operating_hour,,,,,
50,-15.3423,-16.9767,-13.0669,3.9098,2.7281
100,-14.8656,-16.6831,-12.5149,4.1682,2.9577
150,-14.8222,-16.5340,-12.4405,4.0936,2.6561
200,-15.8948,-17.5280,-13.5796,3.9484,2.7809
250,-14.7065,-16.1885,-12.5580,3.6305,2.5334
300,-14.2792,-16.5873,-12.2837,4.3036,3.3394
350,-13.5229,-15.0817,-11.4578,3.6239,2.4263
400,-15.1488,-17.6465,-12.7467,4.8998,3.5590
450,-13.8894,-15.3136,-11.8840,3.4297,2.3291


In [ ]:
### Visualise Durability-Stage Behaviour

The stage-wise median cathode temperature change is visualised together with its interquartile range (IQR).

The median represents the typical temperature change at each durability stage, while the shaded IQR region represents the middle 50% of observations.

In [46]:
plt.figure(figsize=(12, 5))

plt.plot(
    cathode_temp_stage_summary.index,
    cathode_temp_stage_summary["Median"],
    marker="o",
    label="Stage median"
)

plt.fill_between(
    cathode_temp_stage_summary.index,
    cathode_temp_stage_summary["Q1"],
    cathode_temp_stage_summary["Q3"],
    alpha=0.2,
    label="Interquartile range (IQR)"
)

plt.xticks(
    cathode_temp_stage_summary.index,
    rotation=45
)

plt.xlabel("Operating Hour")
plt.ylabel("Cathode Temperature Change (°C)")
plt.title(
    "Cathode Temperature Change Behaviour Across Durability Stages"
)

plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()

plt.show()

<Figure size 1200x500 with 1 Axes>

In [ ]:
### Visualise Stage-Wise Variability

The IQR and standard deviation are compared across durability stages to examine whether the variability of the cathode temperature change changes during the durability test.

In [47]:
plt.figure(figsize=(12, 5))

plt.plot(
    cathode_temp_stage_summary.index,
    cathode_temp_stage_summary["IQR"],
    marker="o",
    label="IQR"
)

plt.plot(
    cathode_temp_stage_summary.index,
    cathode_temp_stage_summary["SD"],
    marker="o",
    label="SD"
)

plt.xticks(
    cathode_temp_stage_summary.index,
    rotation=45
)

plt.xlabel("Operating Hour")
plt.ylabel("Variability (°C)")
plt.title(
    "Stage-Wise Variability of Cathode Temperature Change"
)

plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()

plt.show()

<Figure size 1200x500 with 1 Axes>

In [ ]:
### Association with Voltage

The association between `cathode_temp_diff` and the target variable `voltage` is evaluated using Pearson correlation, Spearman rank correlation, and Mutual Information.

Together, these measures assess linear, monotonic, and more general statistical dependence with voltage.

A strong association would indicate potential predictive relevance, but would not by itself demonstrate degradation sensitivity or causality.

In [48]:
from sklearn.feature_selection import mutual_info_regression

feature = "cathode_temp_diff"
target = "voltage"

pearson_r = df_fe[feature].corr(
    df_fe[target],
    method="pearson"
)

spearman_rho = df_fe[feature].corr(
    df_fe[target],
    method="spearman"
)

mi = mutual_info_regression(
    df_fe[[feature]],
    df_fe[target],
    random_state=42
)[0]

cathode_temp_voltage_association = pd.Series({
    "Pearson": pearson_r,
    "Spearman": spearman_rho,
    "Mutual Information": mi
})

cathode_temp_voltage_association.round(4)

Pearson              -0.0455
Spearman             -0.0293
Mutual Information    0.2255
dtype: float64

In [ ]:
### Parent-Feature Redundancy

The engineered cathode temperature change is compared with its two parent variables:

- `temp_cathode_inlet`
- `temp_cathode_outlet`

Pearson and Spearman associations are calculated to determine whether the engineered feature provides a distinct representation or largely reproduces one of its parent temperature measurements.

Very strong association with a parent variable would indicate potential redundancy and will be considered when deciding whether the feature should be retained for modelling.

In [49]:
parent_features = [
    "temp_cathode_inlet",
    "temp_cathode_outlet"
]

redundancy_results = []

for parent in parent_features:
    redundancy_results.append({
        "Parent Feature": parent,
        "Pearson": df_fe["cathode_temp_diff"].corr(
            df_fe[parent],
            method="pearson"
        ),
        "Spearman": df_fe["cathode_temp_diff"].corr(
            df_fe[parent],
            method="spearman"
        )
    })

cathode_temp_parent_association = pd.DataFrame(
    redundancy_results
)

cathode_temp_parent_association.round(4)

,Parent Feature,Pearson,Spearman
0,temp_cathode_inlet,-0.0542,-0.1197
1,temp_cathode_outlet,0.9643,0.9641


In [ ]:
## Feature 5 — Anode Dew-Point Offset

The anode dew-point offset is defined as:

\[
\Delta T_{dew,a}
=
T_{a,inlet} - T_{dew,a}
\]

using:

- `temp_anode_inlet`
- `temp_anode_dewpoint_water`

This feature represents the temperature margin between the anode inlet gas and its recorded dew-point temperature.

A smaller positive offset indicates that the inlet gas temperature is closer to its dew-point temperature, whereas a larger positive offset indicates a greater temperature margin above the dew point.

The feature is treated as a simple humidification-related indicator. It is not equivalent to relative humidity, membrane water content, or a direct measure of water-management degradation.

Its numerical validity, variability, durability-stage behaviour, voltage association, and redundancy with its parent variables will be evaluated before determining its usefulness for predictive modelling.

In [50]:
# Create anode dew-point offset
df_fe["anode_dewpoint_offset"] = (
    df_fe["temp_anode_inlet"]
    - df_fe["temp_anode_dewpoint_water"]
)

print("Feature created: anode_dewpoint_offset")

Feature created: anode_dewpoint_offset


In [ ]:
### Verify Feature Construction

The first few observations are inspected alongside the anode inlet temperature and anode dew-point temperature to verify:

\[
\Delta T_{dew,a}
=
T_{a,inlet} - T_{dew,a}
\]

The sign and magnitude indicate how far the measured inlet gas temperature is from its recorded dew-point temperature.

In [51]:
df_fe[
    [
        "temp_anode_inlet",
        "temp_anode_dewpoint_water",
        "anode_dewpoint_offset"
    ]
].head()

,temp_anode_inlet,temp_anode_dewpoint_water,anode_dewpoint_offset
0,71.9515,54.4647,17.4867
1,71.9021,54.4647,17.4373
2,71.8897,54.5293,17.3604
3,71.9391,54.4777,17.4615
4,71.8897,54.4777,17.4121


In [ ]:
### Numerical Validity

The engineered anode dew-point offset is checked for missing, infinite, zero, and negative values.

This verifies that feature construction has not introduced invalid values and examines whether the inlet temperature remains above, equal to, or below the recorded dew-point temperature across the complete dataset.

In [52]:
feature = "anode_dewpoint_offset"

anode_dewpoint_validity_summary = pd.Series({
    "count": df_fe[feature].count(),
    "missing": df_fe[feature].isna().sum(),
    "infinite": np.isinf(df_fe[feature]).sum(),
    "zero": (df_fe[feature] == 0).sum(),
    "negative": (df_fe[feature] < 0).sum()
})

anode_dewpoint_validity_summary

count       3629680
missing           0
infinite          0
zero              0
negative          0
dtype: int64

In [ ]:
### Overall Distribution

The overall distribution of the anode dew-point offset is summarised using minimum, maximum, median, interquartile range (IQR), and standard deviation (SD).

These statistics describe the typical temperature-to-dew-point margin and determine whether the engineered feature contains sufficient variation for further investigation.

In [53]:
q1 = df_fe["anode_dewpoint_offset"].quantile(0.25)
q3 = df_fe["anode_dewpoint_offset"].quantile(0.75)

anode_dewpoint_summary = pd.Series({
    "Min": df_fe["anode_dewpoint_offset"].min(),
    "Max": df_fe["anode_dewpoint_offset"].max(),
    "Median": df_fe["anode_dewpoint_offset"].median(),
    "IQR": q3 - q1,
    "SD": df_fe["anode_dewpoint_offset"].std()
})

anode_dewpoint_summary.round(4)

Min       8.3995
Max      22.2689
Median   14.9685
IQR       0.4779
SD        0.4481
dtype: float64

In [ ]:
### Stage-Wise Behaviour

The anode dew-point offset is summarised separately at each durability stage using the median, first quartile (Q1), third quartile (Q3), interquartile range (IQR), and standard deviation (SD).

This allows us to determine whether the typical dew-point margin or its variability changes as operating hours increase.

In [54]:
anode_dewpoint_stage_summary = (
    df_fe
    .groupby("operating_hour")["anode_dewpoint_offset"]
    .agg(
        Median="median",
        Q1=lambda x: x.quantile(0.25),
        Q3=lambda x: x.quantile(0.75),
        SD="std"
    )
)

anode_dewpoint_stage_summary["IQR"] = (
    anode_dewpoint_stage_summary["Q3"]
    - anode_dewpoint_stage_summary["Q1"]
)

anode_dewpoint_stage_summary = anode_dewpoint_stage_summary[
    ["Median", "Q1", "Q3", "IQR", "SD"]
]

anode_dewpoint_stage_summary.round(4)

,Median,Q1,Q3,IQR,SD
operating_hour,,,,,
50,14.9296,14.7255,15.2178,0.4924,0.3987
100,14.9535,14.7554,15.2318,0.4764,0.4024
150,14.9858,14.7749,15.2348,0.4599,0.4022
200,14.9548,14.7574,15.2448,0.4874,0.4423
250,14.9699,14.7619,15.2047,0.4428,0.4229
300,14.9949,14.8055,15.2610,0.4554,0.4332
350,14.9675,14.7837,15.2027,0.4189,0.3978
400,14.9457,14.7257,15.2334,0.5077,0.5187
450,14.9883,14.7800,15.2102,0.4302,0.3948


In [ ]:
### Visualise Stage-Wise Behaviour

The stage-wise median and interquartile range (IQR) are visualised across durability stages.

The median represents the typical anode dew-point offset at each stage, while the shaded IQR region shows the middle 50% of observations. This helps assess whether the humidification-related operating margin shifts or changes in spread as durability progresses.

In [55]:
hours = anode_dewpoint_stage_summary.index

plt.figure(figsize=(14, 5))

plt.plot(
    hours,
    anode_dewpoint_stage_summary["Median"],
    marker="o",
    label="Stage median"
)

plt.fill_between(
    hours,
    anode_dewpoint_stage_summary["Q1"],
    anode_dewpoint_stage_summary["Q3"],
    alpha=0.2,
    label="Interquartile range (IQR)"
)

plt.title("Anode Dew-Point Offset Behaviour Across Durability Stages")
plt.xlabel("Operating Hour")
plt.ylabel("Anode Dew-Point Offset (°C)")

plt.xticks(hours, rotation=45)

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

<Figure size 1400x500 with 1 Axes>

In [ ]:
### Visualise Stage-Wise Variability

The IQR and standard deviation (SD) are compared across durability stages to assess whether the variability of the anode dew-point offset changes over time.

IQR describes the spread of the central 50% of observations, while SD reflects the overall dispersion around the mean.

In [56]:
plt.figure(figsize=(14, 5))

plt.plot(
    hours,
    anode_dewpoint_stage_summary["IQR"],
    marker="o",
    label="IQR"
)

plt.plot(
    hours,
    anode_dewpoint_stage_summary["SD"],
    marker="o",
    label="SD"
)

plt.title("Stage-Wise Variability of Anode Dew-Point Offset")
plt.xlabel("Operating Hour")
plt.ylabel("Variability (°C)")

plt.xticks(hours, rotation=45)

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

<Figure size 1400x500 with 1 Axes>

In [ ]:
### Association with Voltage

The relationship between the anode dew-point offset and voltage is evaluated using Pearson correlation, Spearman rank correlation, and Mutual Information.

Pearson measures linear association, Spearman measures monotonic association, and Mutual Information can identify broader statistical dependence.

These measures determine whether combining anode inlet temperature and dew-point temperature into a single physically interpretable offset reveals voltage-related information not immediately apparent from stage-wise behaviour alone.

In [57]:
from sklearn.feature_selection import mutual_info_regression

feature = "anode_dewpoint_offset"
target = "voltage"

# Pearson correlation
pearson = df_fe[[feature, target]].corr(
    method="pearson"
).loc[feature, target]

# Spearman correlation
spearman = df_fe[[feature, target]].corr(
    method="spearman"
).loc[feature, target]

# Mutual Information
mi = mutual_info_regression(
    df_fe[[feature]],
    df_fe[target],
    random_state=42
)[0]

anode_dewpoint_voltage_association = pd.Series({
    "Pearson": pearson,
    "Spearman": spearman,
    "Mutual Information": mi
})

anode_dewpoint_voltage_association.round(4)

Pearson              -0.3669
Spearman             -0.3408
Mutual Information    0.3056
dtype: float64

In [ ]:
### Parent-Feature Redundancy

Because the anode dew-point offset is constructed directly from anode inlet temperature and anode dew-point temperature, some statistical relationship with its parent variables is expected.

Pearson and Spearman associations are calculated between the engineered feature and each parent variable to determine whether the offset provides a distinct representation or largely reproduces one of its components.

This assessment will be considered together with the feature's voltage association and stage-wise behaviour when deciding whether it should be carried forward.

In [58]:
parent_features = [
    "temp_anode_inlet",
    "temp_anode_dewpoint_water"
]

redundancy_results = []

for parent in parent_features:
    pearson = df_fe[
        ["anode_dewpoint_offset", parent]
    ].corr(method="pearson").iloc[0, 1]

    spearman = df_fe[
        ["anode_dewpoint_offset", parent]
    ].corr(method="spearman").iloc[0, 1]

    redundancy_results.append({
        "Parent Feature": parent,
        "Pearson": pearson,
        "Spearman": spearman
    })

anode_dewpoint_parent_association = pd.DataFrame(
    redundancy_results
)

anode_dewpoint_parent_association.round(4)

,Parent Feature,Pearson,Spearman
0,temp_anode_inlet,0.8891,0.8810
1,temp_anode_dewpoint_water,-0.2862,-0.2874


In [ ]:
## Feature 6 — Cathode Dew-Point Offset

The cathode dew-point offset is defined as:

\[
\Delta T_{dew,c}
=
T_{c,inlet} - T_{dew,c}
\]

using:

- `temp_cathode_inlet`
- `temp_cathode_dewpoint_water`

This feature represents the temperature margin between the cathode inlet air and its recorded dew-point temperature.

A smaller positive offset indicates that the inlet-air temperature is closer to its dew-point temperature, whereas a larger positive offset indicates a greater temperature margin above the dew point.

The feature is treated as a humidification-related operating indicator. It is not equivalent to relative humidity, liquid-water content, flooding, or a direct measure of degradation.

Its numerical validity, variability, durability-stage behaviour, voltage association, and redundancy with its parent variables will be evaluated before determining its usefulness for predictive modelling.

In [59]:
# Create cathode dew-point offset
df_fe["cathode_dewpoint_offset"] = (
    df_fe["temp_cathode_inlet"]
    - df_fe["temp_cathode_dewpoint_water"]
)

print("Feature created: cathode_dewpoint_offset")

Feature created: cathode_dewpoint_offset


In [ ]:
### Verify Feature Construction

The first few observations are inspected alongside the cathode inlet temperature and cathode dew-point temperature to verify:

\[
\Delta T_{dew,c}
=
T_{c,inlet} - T_{dew,c}
\]

The sign and magnitude indicate how far the measured cathode inlet-air temperature is from its recorded dew-point temperature.

In [60]:
df_fe[
    [
        "temp_cathode_inlet",
        "temp_cathode_dewpoint_water",
        "cathode_dewpoint_offset"
    ]
].head()

,temp_cathode_inlet,temp_cathode_dewpoint_water,cathode_dewpoint_offset
0,69.6736,64.4643,5.2093
1,69.6612,64.4390,5.2222
2,69.6859,64.4866,5.1993
3,69.6859,64.4390,5.2469
4,69.6612,64.4895,5.1717


In [ ]:
### Numerical Validity

The engineered cathode dew-point offset is checked for missing, infinite, zero, and negative values.

This verifies that feature construction has not introduced invalid values and examines whether the cathode inlet-air temperature remains above, equal to, or below its recorded dew-point temperature across the complete dataset.

In [61]:
feature = "cathode_dewpoint_offset"

cathode_dewpoint_validity_summary = pd.Series({
    "count": df_fe[feature].count(),
    "missing": df_fe[feature].isna().sum(),
    "infinite": np.isinf(df_fe[feature]).sum(),
    "zero": (df_fe[feature] == 0).sum(),
    "negative": (df_fe[feature] < 0).sum()
})

cathode_dewpoint_validity_summary

count       3629680
missing           0
infinite          0
zero              0
negative          0
dtype: int64

In [ ]:
### Overall Distribution

The overall distribution of the cathode dew-point offset is summarised using minimum, maximum, median, interquartile range (IQR), and standard deviation (SD).

These statistics describe the typical cathode temperature-to-dew-point margin and determine whether the engineered feature contains sufficient variation for further investigation.

In [62]:
q1 = df_fe["cathode_dewpoint_offset"].quantile(0.25)
q3 = df_fe["cathode_dewpoint_offset"].quantile(0.75)

cathode_dewpoint_summary = pd.Series({
    "Min": df_fe["cathode_dewpoint_offset"].min(),
    "Max": df_fe["cathode_dewpoint_offset"].max(),
    "Median": df_fe["cathode_dewpoint_offset"].median(),
    "IQR": q3 - q1,
    "SD": df_fe["cathode_dewpoint_offset"].std()
})

cathode_dewpoint_summary.round(4)

Min       2.7672
Max      13.6248
Median    4.8045
IQR       1.2765
SD        0.8043
dtype: float64

In [ ]:
### Stage-Wise Behaviour

The cathode dew-point offset is summarised separately at each durability stage using the median, first quartile (Q1), third quartile (Q3), interquartile range (IQR), and standard deviation (SD).

This allows us to determine whether the typical cathode dew-point margin or its variability changes as operating hours increase.

In [63]:
cathode_dewpoint_stage_summary = (
    df_fe
    .groupby("operating_hour")["cathode_dewpoint_offset"]
    .agg(
        Median="median",
        Q1=lambda x: x.quantile(0.25),
        Q3=lambda x: x.quantile(0.75),
        SD="std"
    )
)

cathode_dewpoint_stage_summary["IQR"] = (
    cathode_dewpoint_stage_summary["Q3"]
    - cathode_dewpoint_stage_summary["Q1"]
)

cathode_dewpoint_stage_summary = cathode_dewpoint_stage_summary[
    ["Median", "Q1", "Q3", "IQR", "SD"]
]

cathode_dewpoint_stage_summary.round(4)

,Median,Q1,Q3,IQR,SD
operating_hour,,,,,
50,4.7368,4.2770,5.8181,1.5411,0.9464
100,4.7613,4.3251,5.7818,1.4567,0.8923
150,4.7776,4.3354,5.7619,1.4265,0.8753
200,4.8353,4.4728,5.6302,1.1575,0.7344
250,4.8116,4.4301,5.6408,1.2107,0.7723
300,4.8041,4.4329,5.5993,1.1664,0.7650
350,4.8440,4.5144,5.5610,1.0466,0.6875
400,4.8341,4.4353,5.6010,1.1656,0.7366
450,4.7788,4.3785,5.7968,1.4183,0.8527


In [ ]:
### Visualise Stage-Wise Behaviour

The stage-wise median and interquartile range (IQR) are visualised across all durability stages.

The median represents the typical cathode dew-point offset at each stage, while the shaded IQR region represents the central 50% of observations.

This allows changes in both the central operating behaviour and within-stage variability to be examined across operating hours.

In [64]:
hours = cathode_dewpoint_stage_summary.index

plt.figure(figsize=(14, 5))

plt.plot(
    hours,
    cathode_dewpoint_stage_summary["Median"],
    marker="o",
    label="Stage median"
)

plt.fill_between(
    hours,
    cathode_dewpoint_stage_summary["Q1"],
    cathode_dewpoint_stage_summary["Q3"],
    alpha=0.2,
    label="Interquartile range (IQR)"
)

plt.title("Cathode Dew-Point Offset Behaviour Across Durability Stages")
plt.xlabel("Operating Hour")
plt.ylabel("Cathode Dew-Point Offset (°C)")

plt.xticks(hours, rotation=45)

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.show()

<Figure size 1400x500 with 1 Axes>

In [ ]:
### Visualise Stage-Wise Variability

The IQR and standard deviation (SD) are compared across durability stages.

IQR describes variation within the central 50% of observations, while SD reflects the overall spread of the feature.

Examining both measures helps determine whether the variability of the cathode dew-point offset changes systematically with operating hours.

In [65]:
plt.figure(figsize=(14, 5))

plt.plot(
    hours,
    cathode_dewpoint_stage_summary["IQR"],
    marker="o",
    label="IQR"
)

plt.plot(
    hours,
    cathode_dewpoint_stage_summary["SD"],
    marker="o",
    label="SD"
)

plt.title("Stage-Wise Variability of Cathode Dew-Point Offset")
plt.xlabel("Operating Hour")
plt.ylabel("Variability (°C)")

plt.xticks(hours, rotation=45)

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.show()

<Figure size 1400x500 with 1 Axes>

In [ ]:
### Association with Voltage

The relationship between the cathode dew-point offset and stack voltage is evaluated using Pearson correlation, Spearman rank correlation, and Mutual Information.

Pearson measures linear association, Spearman measures monotonic association, and Mutual Information captures broader statistical dependence, including potentially nonlinear relationships.

These results are used to assess whether the engineered cathode dew-point offset contains information relevant to voltage behaviour. Association is not interpreted as evidence of causality or degradation.

In [66]:
feature = "cathode_dewpoint_offset"
target = "voltage"

# Pearson correlation
pearson = df_fe[[feature, target]].corr(
    method="pearson"
).iloc[0, 1]

# Spearman correlation
spearman = df_fe[[feature, target]].corr(
    method="spearman"
).iloc[0, 1]

# Mutual Information
X = df_fe[[feature]]
y = df_fe[target]

mi = mutual_info_regression(
    X,
    y,
    random_state=42
)[0]

cathode_dewpoint_voltage_association = pd.Series({
    "Pearson": pearson,
    "Spearman": spearman,
    "Mutual Information": mi
})

cathode_dewpoint_voltage_association.round(4)

Pearson              -0.7187
Spearman             -0.6385
Mutual Information    0.6185
dtype: float64

In [ ]:
### Parent-Feature Redundancy

Because the cathode dew-point offset is constructed directly from cathode inlet temperature and cathode dew-point temperature, some statistical relationship with its parent variables is expected.

Pearson and Spearman associations are calculated between the engineered feature and each parent variable to assess whether the offset provides a distinct representation or largely reproduces one of its components.

These results will be considered together with the feature's voltage association and stage-wise behaviour before deciding whether it should be carried forward.

In [67]:
parent_features = [
    "temp_cathode_inlet",
    "temp_cathode_dewpoint_water"
]

redundancy_results = []

for parent in parent_features:
    pearson = df_fe[
        ["cathode_dewpoint_offset", parent]
    ].corr(method="pearson").iloc[0, 1]

    spearman = df_fe[
        ["cathode_dewpoint_offset", parent]
    ].corr(method="spearman").iloc[0, 1]

    redundancy_results.append({
        "Parent Feature": parent,
        "Pearson": pearson,
        "Spearman": spearman
    })

cathode_dewpoint_parent_association = pd.DataFrame(
    redundancy_results
)

cathode_dewpoint_parent_association.round(4)

,Parent Feature,Pearson,Spearman
0,temp_cathode_inlet,0.9557,0.9456
1,temp_cathode_dewpoint_water,-0.2342,-0.2112


In [ ]:
# Engineered Feature Review

Six physics-informed candidate features were constructed and empirically evaluated.

Each feature was assessed for:

- numerical validity;
- overall variation;
- durability-stage behaviour;
- association with voltage;
- redundancy with its parent variables;
- and potential usefulness for subsequent predictive modelling.

The purpose of this section is to consolidate the feature-engineering results before preparing the dataset for formal feature selection.

In [68]:
engineered_features = [
    "anode_pressure_diff",
    "cathode_pressure_diff",
    "anode_temp_diff",
    "cathode_temp_diff",
    "anode_dewpoint_offset",
    "cathode_dewpoint_offset"
]

print("Number of engineered features:", len(engineered_features))

print("\nEngineered features:")
for i, feature in enumerate(engineered_features, start=1):
    print(f"{i}. {feature}")

Number of engineered features: 6

Engineered features:
1. anode_pressure_diff
2. cathode_pressure_diff
3. anode_temp_diff
4. cathode_temp_diff
5. anode_dewpoint_offset
6. cathode_dewpoint_offset


In [ ]:
## Preliminary Feature Decisions

The decisions below are not final feature-selection decisions.

They summarise the evidence obtained during feature engineering and identify which variables should be carried forward, treated cautiously, or flagged for potential removal during formal feature selection.

In [69]:
feature_review = pd.DataFrame({
    "Feature": [
        "anode_pressure_diff",
        "cathode_pressure_diff",
        "anode_temp_diff",
        "cathode_temp_diff",
        "anode_dewpoint_offset",
        "cathode_dewpoint_offset"
    ],

    "Voltage_Pearson": [
        -0.6024,
        -0.8570,
        -0.0104,
        -0.0455,
        -0.3669,
        -0.7187
    ],

    "Voltage_Spearman": [
        -0.5305,
        -0.8181,
        -0.0377,
        -0.0293,
        -0.3408,
        -0.6385
    ],

    "Voltage_MI": [
        0.5600,
        1.4286,
        0.2460,
        0.2255,
        0.3056,
        0.6185
    ],

    "Current_Status": [
        "Carry forward",
        "Carry forward",
        "High redundancy / low priority",
        "Conditional / low priority",
        "Carry forward with redundancy flag",
        "Carry forward with redundancy flag"
    ]
})

feature_review

,Feature,Voltage_Pearson,Voltage_Spearman,Voltage_MI,Current_Status
0,anode_pressure_diff,-0.6024,-0.5305,0.5600,Carry forward
1,cathode_pressure_diff,-0.8570,-0.8181,1.4286,Carry forward
2,anode_temp_diff,-0.0104,-0.0377,0.2460,High redundancy / low priority
3,cathode_temp_diff,-0.0455,-0.0293,0.2255,Conditional / low priority
4,anode_dewpoint_offset,-0.3669,-0.3408,0.3056,Carry forward with redundancy flag
5,cathode_dewpoint_offset,-0.7187,-0.6385,0.6185,Carry forward with redundancy flag


In [ ]:
# 10.8 Verify Feature-Engineered Dataset

Before saving the feature-engineered dataset, its final dimensions, engineered columns, missing values, infinite values, and duplicate rows are checked.

This ensures that feature construction has not introduced data-quality problems.

In [70]:
print("Final dataset shape:")
print(df_fe.shape)

print("\nEngineered feature columns:")
print(df_fe[engineered_features].columns.tolist())

print("\nMissing values in engineered features:")
print(df_fe[engineered_features].isna().sum())

print("\nInfinite values in engineered features:")
print(
    np.isinf(
        df_fe[engineered_features]
    ).sum()
)

print("\nDuplicate rows in complete dataset:")
print(df_fe.duplicated().sum())

Final dataset shape:
(3629680, 24)

Engineered feature columns:
['anode_pressure_diff', 'cathode_pressure_diff', 'anode_temp_diff', 'cathode_temp_diff', 'anode_dewpoint_offset', 'cathode_dewpoint_offset']

Missing values in engineered features:
anode_pressure_diff        0
cathode_pressure_diff      0
anode_temp_diff            0
cathode_temp_diff          0
anode_dewpoint_offset      0
cathode_dewpoint_offset    0
dtype: int64

Infinite values in engineered features:
anode_pressure_diff        0
cathode_pressure_diff      0
anode_temp_diff            0
cathode_temp_diff          0
anode_dewpoint_offset      0
cathode_dewpoint_offset    0
dtype: int64

Duplicate rows in complete dataset:
0


In [ ]:
## Preview Final Dataset

The first few observations are displayed to confirm that the engineered features have been appended correctly to the original processed variables.

In [71]:
df_fe.head()

,operating_hour,time,current,voltage,power,pressure_anode_inlet,pressure_anode_outlet,pressure_cathode_inlet,pressure_cathode_outlet,temp_anode_endplate,temp_anode_dewpoint_water,temp_anode_inlet,temp_anode_outlet,temp_cathode_dewpoint_water,temp_cathode_inlet,temp_cathode_outlet,total_anode_stack_flow,total_cathode_stack_flow,anode_pressure_diff,cathode_pressure_diff,anode_temp_diff,cathode_temp_diff,anode_dewpoint_offset,cathode_dewpoint_offset
0,50,1.7610,0.0000,0.9375,0.0000,109.9013,110.3273,109.8001,108.7921,83.1282,54.4647,71.9515,39.4532,64.4643,69.6736,56.3375,0.0700,0.2910,-0.4259,1.0080,-32.4983,-13.3360,17.4867,5.2093
1,50,2.7610,0.0000,0.9375,0.0000,110.1037,110.3273,109.8001,108.8934,83.0788,54.4647,71.9021,39.3870,64.4390,69.6612,56.3249,0.0700,0.2910,-0.2236,0.9067,-32.5150,-13.3363,17.4373,5.2222
2,50,3.7610,0.0000,0.9372,0.0000,110.3060,110.3273,109.8001,108.7921,83.0788,54.5293,71.8897,39.4135,64.4866,69.6859,56.2997,0.0700,0.2910,-0.0212,1.0080,-32.4763,-13.3863,17.3604,5.1993
3,50,4.7610,0.0000,0.9375,0.0000,110.1037,110.3273,109.8001,108.7921,83.1035,54.4777,71.9391,39.3870,64.4390,69.6859,56.3249,0.0700,0.2910,-0.2236,1.0080,-32.5521,-13.3610,17.4615,5.2469
4,50,5.7610,0.0000,0.9372,0.0000,109.9013,110.3273,109.6990,108.8934,83.0911,54.4777,71.8897,39.3738,64.4895,69.6612,56.2492,0.0700,0.2910,-0.4259,0.8056,-32.5159,-13.4121,17.4121,5.1717


In [ ]:
## Confirm Expected Feature Count

The processed dataset originally contained 18 variables. Six engineered features were added in this notebook, so the final feature-engineered dataset is expected to contain 24 columns.

In [72]:
expected_columns = 18 + len(engineered_features)

print("Expected columns:", expected_columns)
print("Actual columns:", df_fe.shape[1])

if df_fe.shape[1] == expected_columns:
    print("Feature count verified successfully.")
else:
    print("Column count differs from expectation. Inspect before saving.")

Expected columns: 24
Actual columns: 24
Feature count verified successfully.


In [ ]:
# 10.9 Save Feature-Engineered Dataset

The complete dataset containing both original variables and engineered candidate features is saved for use in the subsequent feature-selection stage.

No candidate engineered feature is removed at this point. Features flagged for redundancy or weak predictive relevance are retained so that formal feature selection can evaluate them alongside the original variables.

In [73]:
feature_engineered_file = (
    PROCESSED_DATA_DIR / "pemfc_feature_engineered.csv"
)

df_fe.to_csv(
    feature_engineered_file,
    index=False
)

print("Feature-engineered dataset saved successfully.")
print("Saved to:", feature_engineered_file)
print("Final shape:", df_fe.shape)

Feature-engineered dataset saved successfully.
Saved to: C:\Users\usman\Desktop\PEMFC_Dissertation\data\processed\pemfc_feature_engineered.csv
Final shape: (3629680, 24)


In [ ]:
## Verify Saved Dataset

The saved feature-engineered dataset is reloaded and checked to confirm that its dimensions and engineered feature columns were preserved correctly.

In [74]:
df_fe_check = pd.read_csv(feature_engineered_file)

print("Reloaded dataset shape:", df_fe_check.shape)

missing_engineered_features = [
    feature
    for feature in engineered_features
    if feature not in df_fe_check.columns
]

if not missing_engineered_features:
    print("All engineered features verified successfully.")
else:
    print(
        "Missing engineered features:",
        missing_engineered_features
    )

Reloaded dataset shape: (3629680, 24)
All engineered features verified successfully.
